# Generate predicted_masks (Colab, Resumable)

Runs the trained U-Net on `synthetic_dataset/images/` → `synthetic_dataset/predicted_masks/`.

**Resumable**: skips images whose mask already exists. State saved to `/tmp/pipeline_state.json`.

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
if not (REPO / 'src').exists():
    print('Cloning repo...')
    subprocess.run(['git', 'clone', 'https://github.com/ppras/SkylineGeolocation.git', str(REPO)], check=True)
else:
    print(f'Repo already at {REPO}')
os.chdir(REPO)
import sys
sys.path.insert(0, str(REPO))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -qq libgl1-mesa-glx libglib2.0-0 2>/dev/null
!pip install -q segmentation-models-pytorch timm albumentations torchvision

In [ ]:
import os, sys
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
DRIVE = Path('/content/drive/MyDrive')
os.chdir(REPO)
sys.path.insert(0, str(REPO))

for src, dst in [
    (DRIVE / 'sky_segmentation_unet_model.pth', REPO / 'data/sky_segmentation_unet_model.pth'),
    (DRIVE / 'synthetic_dataset', REPO / 'data/synthetic_dataset'),
]:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists():
        os.symlink(src, dst)
        print(f'Linked {src.name}')
    elif dst.exists():
        print(f'{dst.name}: already linked')
    else:
        print(f'MISSING: {src}')

In [ ]:
import torch
from src.segmentation import load_segmentation_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_segmentation_model('data/sky_segmentation_unet_model.pth', device)
print(f'Model loaded on: {device}')

In [ ]:
from src.segmentation import segment_image
from pathlib import Path
from collections import Counter

images_dir = Path('data/synthetic_dataset/images')
out_dir = Path('data/synthetic_dataset/predicted_masks')
out_dir.mkdir(parents=True, exist_ok=True)

image_paths = sorted(images_dir.glob('*.png'))
print(f'Found {len(image_paths)} images')

results = []
skipped = 0
for i, img in enumerate(image_paths):
    mask_path = out_dir / img.name
    if mask_path.exists():
        skipped += 1
        continue
    res = segment_image(model, str(img), str(mask_path), device)
    results.append(res['status'])
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(image_paths)} done (last: {res["status"]})', flush=True)

print(f'Skipped (already done): {skipped}')
print(f'New: {Counter(results)}')
n_done = len(list(out_dir.glob('*.png')))
print(f'predicted_masks: {n_done}/{len(image_paths)} files')

if n_done == len(image_paths):
    import shutil
    shutil.copytree(out_dir, DRIVE / 'synthetic_dataset/predicted_masks', dirs_exist_ok=True)
    print('Copied to Drive')
else:
    print(f'Incomplete. Re-run to continue from {n_done}/{len(image_paths)}')